# CAFA5 Savio Training Results Analysis

Objective: analyze outputs produced by `scripts/savio_train_full_graph.sh` without retraining.

This notebook reads a completed Savio training run directory and generates:
- run configuration and completion status tables
- final train/val/test metrics by GO aspect
- per-epoch learning curves for loss, micro-F1, macro-F1, and Fmax
- validation/test comparison plots
- runtime and GPU assignment summaries
- progress-log parsing from `train.log`

By default it selects the newest run under `graph_cache/training_runs/full_graph_*`. To analyze a specific run, set `CAFA5_ANALYSIS_RUN_DIR` before running the setup cell.

In [ ]:
# Setup and run discovery
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

from IPython.display import display


def find_repo_root(start: Path) -> tuple[Path, str]:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate, 'cwd_search'
    return start, 'cwd_fallback'


def resolve_repo_root() -> tuple[Path, str]:
    configured = os.environ.get('CAFA5_REPO_ROOT')
    if configured:
        return Path(configured).expanduser().resolve(), 'env:CAFA5_REPO_ROOT'
    return find_repo_root(Path.cwd())


def newest_run_dir(training_runs_dir: Path, pattern: str) -> Path:
    candidates = [path for path in training_runs_dir.glob(pattern) if path.is_dir()]
    if not candidates:
        raise FileNotFoundError(f'No run directories matched {training_runs_dir / pattern}')
    return max(candidates, key=lambda path: path.stat().st_mtime)


REPO_ROOT, REPO_ROOT_SOURCE = resolve_repo_root()
SERVER_USER = os.environ.get('USER', 'bensonli')
RUN_ROOT = Path(os.environ.get('CAFA5_RUN_ROOT', f'/global/scratch/users/{SERVER_USER}/cafa5_outputs'))
GRAPH_CACHE_DIR = RUN_ROOT / 'graph_cache'
TRAINING_RUNS_DIR = GRAPH_CACHE_DIR / 'training_runs'
RUN_GLOB = os.environ.get('CAFA5_ANALYSIS_RUN_GLOB', 'full_graph_*')
configured_run_dir = os.environ.get('CAFA5_ANALYSIS_RUN_DIR')
RUN_DIR = Path(configured_run_dir).expanduser() if configured_run_dir else newest_run_dir(TRAINING_RUNS_DIR, RUN_GLOB)
RUN_DIR = RUN_DIR.resolve()
FIGURE_DIR = RUN_DIR / 'analysis_figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

config = {
    'repo_root': str(REPO_ROOT),
    'repo_root_source': REPO_ROOT_SOURCE,
    'run_root': str(RUN_ROOT),
    'training_runs_dir': str(TRAINING_RUNS_DIR),
    'run_glob': RUN_GLOB,
    'run_dir': str(RUN_DIR),
    'figure_dir': str(FIGURE_DIR),
}
print(json.dumps(config, indent=2))

required_paths = [RUN_DIR]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing required paths: {missing_paths}')

## Load Run Files

`savio_train_full_graph.sh` writes top-level metadata plus one subdirectory per aspect (`bpo/`, `cco/`, `mfo/`). This section loads all of those artifacts into DataFrames.

In [ ]:
# Load config, git metadata, and result summaries.
def read_json_if_exists(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def read_text_if_exists(path: Path, default: str = '') -> str:
    return path.read_text(encoding='utf-8', errors='replace').strip() if path.exists() else default


run_config = read_json_if_exists(RUN_DIR / 'run_config.json', default={})
results_rows = read_json_if_exists(RUN_DIR / 'results_summary.json', default=[])
if not results_rows and (RUN_DIR / 'results_summary.tsv').exists() and pd is not None:
    results_rows = pd.read_csv(RUN_DIR / 'results_summary.tsv', sep='\t').to_dict('records')

metadata = {
    'git_branch': read_text_if_exists(RUN_DIR / 'git_branch.txt'),
    'git_commit': read_text_if_exists(RUN_DIR / 'git_commit.txt'),
    'run_config_exists': (RUN_DIR / 'run_config.json').exists(),
    'results_summary_json_exists': (RUN_DIR / 'results_summary.json').exists(),
    'results_summary_tsv_exists': (RUN_DIR / 'results_summary.tsv').exists(),
    'driver_log_exists': (RUN_DIR / 'driver.log').exists(),
}
print(json.dumps(metadata, indent=2))

if not results_rows:
    raise FileNotFoundError(f'No results summary found in {RUN_DIR}. Expected results_summary.json or results_summary.tsv.')

if pd is not None:
    results_df = pd.DataFrame(results_rows)
    display(results_df)
else:
    print(json.dumps(results_rows, indent=2))

In [ ]:
# Load per-aspect run_result.json and summary.json histories.
def aspect_dirs(run_dir: Path) -> list[Path]:
    return sorted(path for path in run_dir.iterdir() if path.is_dir() and (path / 'run_result.json').exists())


run_result_rows = []
epoch_rows = []
for aspect_dir in aspect_dirs(RUN_DIR):
    run_result = read_json_if_exists(aspect_dir / 'run_result.json', default={})
    if not run_result:
        continue
    aspect = str(run_result.get('aspect') or aspect_dir.name.upper())
    run_result_rows.append({
        'aspect': aspect,
        'status': run_result.get('status'),
        'status_code': run_result.get('status_code'),
        'gpu_id': run_result.get('gpu_id'),
        'started_at': run_result.get('started_at'),
        'finished_at': run_result.get('finished_at'),
        'checkpoint_dir': run_result.get('checkpoint_dir'),
        'summary_path': run_result.get('summary_path'),
        'best_val_loss': run_result.get('best_val_loss'),
        'best_checkpoint_path': run_result.get('best_checkpoint_path'),
    })
    summary_path = aspect_dir / 'summary.json'
    summary = read_json_if_exists(summary_path, default={})
    for record in summary.get('history', []):
        for split_name in ('train', 'val', 'test'):
            metrics = record.get(split_name, {}) or {}
            row = {
                'aspect': aspect,
                'split': split_name,
                'epoch': record.get('epoch'),
                'epoch_seconds': record.get('epoch_seconds'),
            }
            for key in (
                'loss',
                'micro_precision',
                'micro_recall',
                'micro_f1',
                'macro_precision',
                'macro_recall',
                'macro_f1',
                'macro_f1_positive_labels',
                'fmax',
                'fmax_threshold',
                'fmax_precision',
                'fmax_recall',
                'graphs',
            ):
                row[key] = metrics.get(key)
            epoch_rows.append(row)

if pd is None:
    print(json.dumps({'run_results': run_result_rows, 'epoch_rows_preview': epoch_rows[:20]}, indent=2))
else:
    run_result_df = pd.DataFrame(run_result_rows)
    epoch_df = pd.DataFrame(epoch_rows)
    display(run_result_df)
    display(epoch_df.head(20))
    print('epoch rows =', len(epoch_df))

## Final Metrics

The top-level `results_summary.json` contains final-epoch metrics for each aspect. Use this for quick comparison across `BPO`, `CCO`, and `MFO`.

In [ ]:
# Final metric table focused on validation and test performance.
if pd is None:
    raise ImportError('pandas is required for tabular analysis')

metric_cols = [
    'aspect',
    'status',
    'gpu_id',
    'final_epoch',
    'best_val_loss',
    'val_loss',
    'val_micro_f1',
    'val_macro_f1',
    'val_fmax',
    'val_fmax_threshold',
    'test_loss',
    'test_micro_f1',
    'test_macro_f1',
    'test_fmax',
    'test_fmax_threshold',
    'checkpoint_dir',
]
available_cols = [col for col in metric_cols if col in results_df.columns]
final_metrics_df = results_df[available_cols].copy()
for col in final_metrics_df.columns:
    if col.endswith(('loss', 'f1', 'fmax', 'threshold')) or col == 'best_val_loss':
        final_metrics_df[col] = pd.to_numeric(final_metrics_df[col], errors='coerce')
display(final_metrics_df.sort_values(['status', 'aspect']))

summary_out = FIGURE_DIR / 'final_metrics.tsv'
final_metrics_df.to_csv(summary_out, sep='\t', index=False)
print('wrote', summary_out)

In [ ]:
# Plot final val/test metrics by aspect.
if pd is None or plt is None:
    raise ImportError('pandas and matplotlib are required for plotting')

plot_metric_cols = ['micro_f1', 'macro_f1', 'fmax']
plot_rows = []
for _, row in results_df.iterrows():
    for split_name in ('val', 'test'):
        for metric in plot_metric_cols:
            col = f'{split_name}_{metric}'
            if col in results_df.columns:
                plot_rows.append({'aspect': row.get('aspect'), 'split': split_name, 'metric': metric, 'value': row.get(col)})
plot_df = pd.DataFrame(plot_rows)
plot_df['value'] = pd.to_numeric(plot_df['value'], errors='coerce')

fig, axes = plt.subplots(1, len(plot_metric_cols), figsize=(5 * len(plot_metric_cols), 4), squeeze=False)
for ax, metric in zip(axes[0], plot_metric_cols):
    pivot = plot_df[plot_df['metric'] == metric].pivot(index='aspect', columns='split', values='value')
    pivot.plot(kind='bar', ax=ax)
    ax.set_title(f'Final {metric}')
    ax.set_xlabel('aspect')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
figure_path = FIGURE_DIR / 'final_val_test_metrics.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('wrote', figure_path)

## Learning Curves

Per-epoch curves help detect underfitting, overfitting, and unstable validation behavior. These metrics are unweighted; CAFA IA-weighted `wFmax` still requires prediction TSV export and `cafaeval`.

In [ ]:
# Plot per-epoch loss and score curves.
if pd is None or plt is None:
    raise ImportError('pandas and matplotlib are required for plotting')
if epoch_df.empty:
    raise ValueError('epoch_df is empty. No per-epoch summary.json histories were found.')

aspects = list(epoch_df['aspect'].dropna().unique())
metrics_to_plot = ['loss', 'micro_f1', 'macro_f1', 'fmax']
fig, axes = plt.subplots(len(metrics_to_plot), len(aspects), figsize=(5 * len(aspects), 3.2 * len(metrics_to_plot)), squeeze=False)
for col_index, aspect in enumerate(aspects):
    aspect_df = epoch_df[epoch_df['aspect'] == aspect]
    for row_index, metric in enumerate(metrics_to_plot):
        ax = axes[row_index][col_index]
        for split_name in ('train', 'val', 'test'):
            split_df = aspect_df[aspect_df['split'] == split_name].sort_values('epoch')
            if split_df.empty or metric not in split_df.columns:
                continue
            ax.plot(split_df['epoch'], pd.to_numeric(split_df[metric], errors='coerce'), marker='o', label=split_name)
        ax.set_title(f'{aspect} {metric}')
        ax.set_xlabel('epoch')
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.3)
        if row_index == 0 and col_index == 0:
            ax.legend()
plt.tight_layout()
figure_path = FIGURE_DIR / 'learning_curves.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('wrote', figure_path)

In [ ]:
# Plot validation/test Fmax threshold selected per epoch.
if pd is None or plt is None:
    raise ImportError('pandas and matplotlib are required for plotting')

threshold_df = epoch_df[epoch_df['split'].isin(['val', 'test'])].copy()
threshold_df['fmax_threshold'] = pd.to_numeric(threshold_df['fmax_threshold'], errors='coerce')
fig, ax = plt.subplots(figsize=(8, 4))
for (aspect, split_name), group in threshold_df.groupby(['aspect', 'split']):
    group = group.sort_values('epoch')
    ax.plot(group['epoch'], group['fmax_threshold'], marker='o', label=f'{aspect} {split_name}')
ax.set_title('Fmax threshold by epoch')
ax.set_xlabel('epoch')
ax.set_ylabel('threshold')
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
figure_path = FIGURE_DIR / 'fmax_thresholds.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('wrote', figure_path)

## Runtime and Progress Logs

This section parses `train.log` progress lines and compares runtime across aspects. It is useful for spotting a stalled dataloader or an imbalanced aspect run.

In [ ]:
# Runtime summary by aspect.
if pd is None:
    raise ImportError('pandas is required for runtime analysis')

runtime_df = run_result_df.copy() if 'run_result_df' in globals() else pd.DataFrame(run_result_rows)
for col in ('started_at', 'finished_at'):
    runtime_df[col] = pd.to_datetime(runtime_df[col], errors='coerce')
runtime_df['duration_minutes'] = (runtime_df['finished_at'] - runtime_df['started_at']).dt.total_seconds() / 60
runtime_df['gpu_id'] = runtime_df['gpu_id'].astype(str)
display(runtime_df[['aspect', 'status', 'status_code', 'gpu_id', 'duration_minutes', 'checkpoint_dir']])

fig, ax = plt.subplots(figsize=(7, 4))
runtime_df.plot(kind='bar', x='aspect', y='duration_minutes', ax=ax, legend=False)
ax.set_title('Training duration by aspect')
ax.set_xlabel('aspect')
ax.set_ylabel('minutes')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
figure_path = FIGURE_DIR / 'runtime_by_aspect.png'
plt.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('wrote', figure_path)

In [ ]:
# Parse progress lines from per-aspect train.log files.
progress_pattern = re.compile(
    r"\[progress\]\s+epoch=(?P<epoch>\d+)\s+(?P<label>.+?)\s+batch=(?P<batch>\d+)/(?:\?|(?P<total_batches>\d+))\s+graphs=(?P<graphs>\d+)\s+loss=(?P<loss>[-+0-9.eE]+)"
)
progress_rows = []
for aspect_dir in aspect_dirs(RUN_DIR):
    aspect = aspect_dir.name.upper()
    log_path = aspect_dir / 'train.log'
    if not log_path.exists():
        continue
    for line in log_path.read_text(encoding='utf-8', errors='replace').splitlines():
        match = progress_pattern.search(line)
        if not match:
            continue
        payload = match.groupdict()
        label_parts = payload['label'].split()
        split_name = label_parts[-1] if label_parts else None
        progress_rows.append({
            'aspect': aspect,
            'epoch': int(payload['epoch']),
            'label': payload['label'],
            'split': split_name,
            'batch': int(payload['batch']),
            'total_batches': int(payload['total_batches']) if payload.get('total_batches') else None,
            'graphs': int(payload['graphs']),
            'loss': float(payload['loss']),
        })

progress_df = pd.DataFrame(progress_rows) if pd is not None else None
if progress_df is None or progress_df.empty:
    print('No progress lines found. This is expected if PROGRESS_MODE was not log, or logs are unavailable.')
else:
    display(progress_df.tail(20))
    fig, ax = plt.subplots(figsize=(9, 4))
    for (aspect, split_name), group in progress_df.groupby(['aspect', 'split']):
        if split_name != 'train':
            continue
        group = group.sort_values(['epoch', 'batch'])
        ax.plot(range(len(group)), group['loss'], label=f'{aspect} train')
    ax.set_title('Progress-log running training loss')
    ax.set_xlabel('progress checkpoint')
    ax.set_ylabel('running loss')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    figure_path = FIGURE_DIR / 'progress_running_loss.png'
    plt.savefig(figure_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('wrote', figure_path)

## Evaluation Inputs

The Savio script links evaluation inputs under `eval_inputs/`. This notebook only verifies those files. It does not compute CAFA official IA-weighted `wFmax` until prediction TSV export exists.

In [ ]:
# Inspect staged CAFA-evaluator inputs.
eval_dir = RUN_DIR / 'eval_inputs'
eval_rows = []
for name in ('IA.txt', 'go-basic.obo', 'test_terms.tsv', 'cafaeval_command_template.sh', 'README.txt'):
    path = eval_dir / name
    eval_rows.append({
        'name': name,
        'path': str(path),
        'exists': path.exists(),
        'is_symlink': path.is_symlink(),
        'target': str(path.resolve()) if path.exists() else None,
        'size_bytes': path.stat().st_size if path.exists() else None,
    })
if pd is not None:
    display(pd.DataFrame(eval_rows))
else:
    print(json.dumps(eval_rows, indent=2))

command_template = eval_dir / 'cafaeval_command_template.sh'
if command_template.exists():
    print(command_template.read_text(encoding='utf-8'))

## Interpretation Notes

Use this section after running the notebook:

- Compare `val_fmax` and `test_fmax` across aspects.
- Check whether `macro_f1` is much lower than `micro_f1`; if so, frequent GO terms dominate performance.
- Check whether validation loss improves while Fmax stalls; this suggests thresholding or label imbalance issues.
- Check `duration_minutes` and progress logs for dataloader bottlenecks.
- Next implementation step: export prediction TSVs and run `cafaeval` with `IA.txt` for CAFA-style `wFmax`.